# Spatial Sampling and Aliasing
## Tutorial 2: Spatial Frequency, Visible Region, and Aliasing Effects

This notebook explores **spatial sampling** by a Uniform Linear Array, drawing a tight analogy with temporal sampling (Nyquist theorem).  We cover:

1. **Spatial Frequency** – how angle maps to frequency
2. **Visible Region** – the unambiguous range of spatial frequencies
3. **Spatial Aliasing** – what happens when $d > \lambda/2$
4. **Effect of Spacing on Beamwidth and Ambiguities**
5. **Grating Lobes**

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys, os

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from doa_methods.array_processing import UniformLinearArray, SignalModel

plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.size'] = 12

print("Setup complete.")

## 1. The Spatial Frequency Concept

For a ULA with inter-element spacing $d$ (in wavelengths), the received phase shift between adjacent elements for a plane wave at DOA $\theta$ is:

$$\omega = 2\pi \frac{d}{\lambda} \sin\theta \quad [\text{rad/element}]$$

This is the **spatial frequency**.  It plays the same role as temporal frequency $\omega_t = 2\pi f$ in time-domain sampling.

### Analogy with temporal sampling

| Temporal domain | Spatial domain |
|---|---|
| Time $t$ | Element index $m$ |
| Temporal frequency $f$ | Spatial frequency $u = (d/\lambda)\sin\theta$ |
| Sampling rate $f_s$ | Element spacing $d$ |
| Nyquist: $f_s \geq 2 f_{\max}$ | Nyquist: $d \leq \lambda/2$ |

In [ ]:
array = UniformLinearArray(M=8, d=0.5)

theta_deg = np.linspace(-90, 90, 501)
theta_rad = np.deg2rad(theta_deg)

# Spatial frequency u = (d/λ)sin(θ)  (normalised, no 2π)
u = array.d * np.sin(theta_rad)

plt.figure(figsize=(10, 5))
plt.plot(theta_deg, u, 'b-', lw=2)
plt.axhline( 0.5, color='r', ls='--', label='Nyquist limit +0.5')
plt.axhline(-0.5, color='r', ls='--', label='Nyquist limit -0.5')
plt.fill_between(theta_deg, -0.5, 0.5, alpha=0.1, color='green', label='Unambiguous region')
plt.xlabel('DOA θ (degrees)')
plt.ylabel('Normalised spatial frequency u = (d/λ)sin θ')
plt.title('Spatial Frequency vs DOA  (d = 0.5λ)')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"At θ = ±90°: u = ±{array.d:.2f}  ({'within' if array.d <= 0.5 else 'outside'} ±0.5)")

## 2. The Visible Region

The **visible region** is the range of spatial frequencies produced by real angles $\theta \in [-90°, 90°]$:

$$u \in \left[-\frac{d}{\lambda},\; \frac{d}{\lambda}\right]$$

For $d = \lambda/2$: visible region $\in [-0.5, +0.5]$ — exactly the Nyquist interval, so **no aliasing**.

For $d > \lambda/2$: the visible region **extends beyond** $\pm 0.5$, causing aliasing (grating lobes).

In [ ]:
spacings = [0.3, 0.5, 0.8, 1.2]
fig, axes = plt.subplots(1, len(spacings), figsize=(16, 5), sharey=True)

theta_fine = np.linspace(-np.pi/2, np.pi/2, 1801)

for ax, d in zip(axes, spacings):
    arr = UniformLinearArray(M=8, d=d)
    u_vals = d * np.sin(theta_fine)

    ax.plot(np.rad2deg(theta_fine), u_vals, 'b-', lw=2)
    ax.axhline( 0.5, color='r', ls='--', alpha=0.7)
    ax.axhline(-0.5, color='r', ls='--', alpha=0.7)
    ax.fill_between(np.rad2deg(theta_fine), -0.5, 0.5,
                    alpha=0.12, color='green')
    ax.set_title(f'd = {d}λ')
    ax.set_xlabel('θ (°)')
    ax.grid(True, alpha=0.3)
    if d > 0.5:
        ax.text(0.5, 0.97, '⚠ Aliasing', transform=ax.transAxes,
                ha='center', va='top', color='red', fontweight='bold',
                bbox=dict(facecolor='yellow', alpha=0.6, boxstyle='round'))

axes[0].set_ylabel('u = (d/λ)sin θ')
plt.suptitle('Visible Region for Different Inter-Element Spacings', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 3. Spatial Aliasing and Grating Lobes

When $d > \lambda/2$, multiple physical angles map to the same spatial frequency — ambiguities arise.

**Grating lobe condition**: grating lobes appear at angles $\theta_g$ satisfying:

$$\frac{d}{\lambda}(\sin\theta - \sin\theta_g) = n, \quad n = \pm1, \pm2, \ldots$$

For $d = \lambda$, a source at $\theta_0 = 0°$ also produces a grating lobe at $\theta_g = 0°$ (trivial) and at $\sin\theta_g = \sin\theta_0 - n/d$, i.e. at $\pm 90°$ for $n=\pm1$.

In [ ]:
def array_factor(M, d, theta_scan_rad, theta_grid_rad):
    """Normalised array factor when steering to theta_scan."""
    sv_scan = np.exp(-1j * 2*np.pi*d * np.arange(M) * np.sin(theta_scan_rad))
    manifold = np.exp(-1j * 2*np.pi*d *
                      np.outer(np.arange(M), np.sin(theta_grid_rad)))
    af = np.abs(sv_scan.conj() @ manifold)**2 / M**2
    return af

theta_steer = np.deg2rad(0)          # steering to broadside
theta_grid  = np.linspace(-np.pi/2, np.pi/2, 1801)
M = 8

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, d in zip(axes.flatten(), [0.3, 0.5, 0.8, 1.2]):
    af = array_factor(M, d, theta_steer, theta_grid)
    ax.plot(np.rad2deg(theta_grid), 10*np.log10(af + 1e-10), lw=2)
    ax.axvline(np.rad2deg(theta_steer), color='green', ls='--',
               label=f'Steering {np.rad2deg(theta_steer):.0f}°')
    ax.set_ylim(-40, 3)
    ax.set_title(f'd = {d}λ  (M={M})')
    ax.set_xlabel('θ (°)'); ax.set_ylabel('AF (dB)')
    ax.grid(True, alpha=0.3); ax.legend(fontsize=9)

plt.suptitle('Array Factor: Grating Lobes for d > λ/2', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 4. Beamwidth vs Aperture

Larger aperture → narrower mainlobe.  The 3-dB beamwidth of a ULA steered to broadside is approximately:

$$\Delta\theta_{3\text{dB}} \approx \frac{0.886}{Md/\lambda} \text{ rad} \approx \frac{0.886\lambda}{L}$$

where $L = (M-1)d$ is the aperture length.

In [ ]:
d = 0.5
theta_grid = np.linspace(-np.pi/2, np.pi/2, 3601)
theta_steer = 0.0

fig, ax = plt.subplots(figsize=(12, 6))
colors = plt.cm.viridis(np.linspace(0, 1, 5))

for col, M in zip(colors, [4, 8, 16, 32, 64]):
    af = array_factor(M, d, theta_steer, theta_grid)
    bw_theory = np.rad2deg(0.886 / (M * d))
    ax.plot(np.rad2deg(theta_grid), 10*np.log10(af + 1e-10),
            label=f'M={M}  BW≈{bw_theory:.1f}°', color=col, lw=1.8)

ax.axhline(-3, color='gray', ls=':', label='–3 dB')
ax.set_ylim(-40, 3)
ax.set_xlabel('θ (°)'); ax.set_ylabel('Array Factor (dB)')
ax.set_title('Beamwidth vs Number of Elements  (d=0.5λ)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Summary

| Spacing | Visible region | Ambiguities | Beamwidth |
|---|---|---|---|
| $d < \lambda/2$ | $< [-0.5, +0.5]$ | None | Wide |
| $d = \lambda/2$ | $= [-0.5, +0.5]$ | None | Standard |
| $d > \lambda/2$ | $> [-0.5, +0.5]$ | Grating lobes | Narrow (but ambiguous) |

**Key rule**: always use $d = \lambda/2$ unless you have a specific reason and a strategy to resolve ambiguities.

## Exercises
1. For $M = 16, d = 0.5\lambda$, compute the theoretical 3-dB beamwidth and verify it numerically.
2. Show that for $d = 1\lambda$ and a source at $\theta = 30°$, there exists a second angle (grating lobe) that produces an identical array response.
3. Plot the visible region for $d = 0.75\lambda$ and identify the ambiguous angle pairs for $\theta_0 = \pm 20°$.